In [35]:
import pandas as pd
import os
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from openai import OpenAI
from pydantic import BaseModel
import json

# Load API key
with open("key.txt", "r") as f:
    api_key = f.read().strip()

client = OpenAI(api_key=api_key)


In [36]:
import re

# Arabic tashkeel unicode range
TASHKEEL_REGEX = re.compile(r'[\u0617-\u061A\u064B-\u0652]')

def remove_tashkeel(text):
    if not isinstance(text, str):
        return text
    return re.sub(TASHKEEL_REGEX, '', text)


In [37]:
# Load Datasets
df_pos = pd.read_csv("majaz_dataset.csv")
df_neg = pd.read_csv("non_majaz_random_sample.csv")

TEXT_COL = "البيت_الكامل"

# Ensure we only keep the text column to avoid schema errors
df_pos = df_pos[[TEXT_COL]].copy()
df_neg = df_neg[[TEXT_COL]].copy()

# Assign Labels
df_pos["label"] = 1
df_neg["label"] = 0

# Merge and Clean
df = pd.concat([df_pos, df_neg], ignore_index=True)
df = df.dropna(subset=[TEXT_COL])
df[TEXT_COL] = df[TEXT_COL].apply(remove_tashkeel)

# Split Data (Stratified to maintain class balance)
X_train, X_test, y_train, y_test = train_test_split(
    df[TEXT_COL].values,
    df["label"].values,
    test_size=0.10, 
    stratify=df["label"],
    random_state=42
)

print(f"Training size: {len(X_train)}")
print(f"Testing size: {len(X_test)}")

Training size: 1434
Testing size: 160


In [38]:
# 1. Define the Strict Schema
class VerseClassification(BaseModel):
    id: int
    reasoning: str  # We ask the model to explain first (Chain of Thought)
    is_majaz: int   # 0 or 1

class BatchResponse(BaseModel):
    results: list[VerseClassification]

# 2. The Optimized System Prompt
SYSTEM_PROMPT = """
أنت عالم ضليع في البلاغة العربية والنقد الأدبي. مهمتك هي تصنيف الأبيات الشعرية بدقة متناهية.

المهمة:
تحديد ما إذا كان البيت الشعري يحتوي على "مجاز" أم أنه تعبير لا يحتوي على مجاز

قواعد التصنيف:
(1) مجاز:

(0) ليس مجازاً (حقيقي):

أعطِ الحكم بناءً على المعنى البلاغي الدقيق.
"""

In [ ]:
def get_predictions_batch(verses, start_index):
    # Format verses with IDs to track them: "ID 0: text..."
    formatted_verses = "\n".join([f"ID {i+start_index}: {v}" for i, v in enumerate(verses)])

    user_content = f"""
    قم بتحليل الأبيات التالية وتصنيفها.
    
    الأبيات:
    {formatted_verses}
    """

    try:
        completion = client.beta.chat.completions.parse(
            model="gpt-4.1-2025-04-14",  # Best model for Structured Outputs
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_content},
            ],
            response_format=BatchResponse,
            temperature=0.0, # Deterministic ensures consistency
        )
        return completion.choices[0].message.parsed.results
    except Exception as e:
        print(f"Error in batch starting at index {start_index}: {e}")
        return []

In [40]:
BATCH_SIZE = 1
predictions_map = {} # Using a dict to safely map ID -> Prediction

print(f"Starting prediction on {len(X_test)} verses...")

for i in tqdm(range(0, len(X_test), BATCH_SIZE)):
    batch_verses = X_test[i : i + BATCH_SIZE]
    
    # Call API
    results = get_predictions_batch(batch_verses, start_index=i)
    
    # Store results
    for res in results:
        predictions_map[res.id] = {
            "pred": res.is_majaz,
            "reason": res.reasoning
        }

print("Prediction complete.")

Starting prediction on 160 verses...


100%|██████████| 160/160 [03:56<00:00,  1.48s/it]

Prediction complete.


In [41]:
# Reconstruct predictions in the exact order of X_test
final_preds = []
final_reasons = []

for i in range(len(X_test)):
    if i in predictions_map:
        final_preds.append(predictions_map[i]["pred"])
        final_reasons.append(predictions_map[i]["reason"])
    else:
        # Fallback for failed API calls
        final_preds.append(0) 
        final_reasons.append("API Error / Missing")

# Metrics
print(f"Accuracy: {accuracy_score(y_test, final_preds):.4f}")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, final_preds))
print("\nClassification Report:")
print(classification_report(y_test, final_preds, target_names=["Non-Majaz (0)", "Majaz (1)"]))

# Save to CSV
results_df = pd.DataFrame({
    "verse": X_test,
    "y_true": y_test,
    "y_pred": final_preds,
    "reasoning": final_reasons
})

out_path = "gpt_majaz_results.csv"
results_df.to_csv(out_path, index=False, encoding="utf-8-sig")
print(f"Results saved to {out_path}")

# Display first few errors for inspection
print("\nSample Errors:")
print(results_df[results_df["y_true"] != results_df["y_pred"]].head(3))

Accuracy: 0.5062

Confusion Matrix:
[[ 3 73]
 [ 6 78]]

Classification Report:
               precision    recall  f1-score   support

Non-Majaz (0)       0.33      0.04      0.07        76
    Majaz (1)       0.52      0.93      0.66        84

     accuracy                           0.51       160
    macro avg       0.42      0.48      0.37       160
 weighted avg       0.43      0.51      0.38       160

Results saved to gpt_majaz_results.csv

Sample Errors:
                                               verse  y_true  y_pred  \
2    والبطن ذو عكن، لطيف طيه، والإتب تنفجه بثدي مقعد       0       1   
3  لئن كنت قد بلغت عني خيانة، لمبلغك الواشي أغش و...       0       1   
5  فأحيي هداك الله نفسا مريضة طويلا بكم تهيامها و...       0       1   

                                           reasoning  
2  البيت يحتوي على تعبير مجازي حيث يتم استخدام 'ا...  
3  البيت يحتوي على مجاز، حيث يتم استخدام كلمة 'خي...  
5  البيت يحتوي على تعبير مجازي حيث يُستخدم 'أحيي'...  
